In [0]:
CATALOG      = "clutchlytics"
BRONZE_TABLE = f"{CATALOG}.bronze.raw_nhl_teams"
SILVER_TABLE = f"{CATALOG}.silver.dimTeams"
 
# Sport/league context — update these when running for NFL, NBA, etc.
SPORT  = "hockey"
LEAGUE = "nhl"
 
print(f"Source : {BRONZE_TABLE}")
print(f"Target : {SILVER_TABLE}")
print(f"Sport  : {SPORT}")
print(f"League : {LEAGUE}")

In [0]:
# ── READ BRONZE ───────────────────────────────────────────────────────────────
 
bronze_df = spark.table(BRONZE_TABLE)
print(f"Bronze rows: {bronze_df.count()}")
bronze_df.printSchema()

In [0]:
# ── TRANSFORM ────────────────────────────────────────────────────────────────
# - Add sport + league columns
# - Cast types
# - Drop uid, slug, is_all_star
# - clutch_team_id assigned after dedup check below
 
from pyspark.sql import functions as F
 
transformed_df = (
    bronze_df
    .select(
        # ── ESPN native ID — kept for Bronze joins ──
        F.col("team_id").cast("integer").alias("team_id"),
 
        # ── Sport + league context ──
        F.lit(SPORT).alias("sport"),
        F.lit(LEAGUE).alias("league"),
 
        # ── Identity ──
        F.col("abbreviation"),
        F.col("display_name"),
        F.col("short_name"),
        F.col("name"),
        F.col("nickname"),
        F.col("location"),
 
        # ── Branding (kept per spec) ──
        F.col("color"),
        F.col("alternate_color"),
        F.col("logo_href"),
 
        # ── Venue ──
        F.col("venue_id"),
        F.col("venue_name"),
        F.col("venue_city"),
        F.col("venue_state"),
 
        # ── Flags ──
        F.col("is_active").cast("boolean").alias("is_active"),
 
        # ── Metadata ──
        F.col("season").cast("integer").alias("season"),
        F.col("ingested_at"),
        F.lit("bronze.raw_nhl_teams").alias("source_table"),
    )
    .dropDuplicates(["team_id", "league"])
)
 
print(f"Transformed rows: {transformed_df.count()}")

In [0]:
# ── ASSIGN clutch_team_id ─────────────────────────────────────────────────────
# Surrogate PK — sequential row number in order added.
# If table already exists, continue the sequence from the current max.
# New league loads always append — never overwrite existing rows.
 
from pyspark.sql.window import Window
 
table_exists = spark.catalog.tableExists(SILVER_TABLE)
 
if table_exists:
    # Get current max clutch_team_id to continue sequence
    max_id = spark.sql(f"""
        SELECT COALESCE(MAX(clutch_team_id), 0) AS max_id
        FROM {SILVER_TABLE}
    """).collect()[0]["max_id"]
 
    # Check if this league already exists — if so, UPDATE not INSERT
    existing_leagues = spark.sql(f"""
        SELECT DISTINCT league FROM {SILVER_TABLE}
    """).rdd.flatMap(lambda x: x).collect()
 
    print(f"Table exists. Current max clutch_team_id: {max_id}")
    print(f"Existing leagues: {existing_leagues}")
 
else:
    max_id = 0
    existing_leagues = []
    print("Table does not exist yet — first load.")
 
# Assign sequential IDs continuing from max_id
window_spec = Window.orderBy("team_id")
 
silver_df = (
    transformed_df
    .withColumn(
        "clutch_team_id",
        F.row_number().over(window_spec) + max_id
    )
    # Reorder columns — clutch_team_id first
    .select(
        "clutch_team_id",
        "team_id",
        "sport",
        "league",
        "abbreviation",
        "display_name",
        "short_name",
        "name",
        "nickname",
        "location",
        "color",
        "alternate_color",
        "logo_href",
        "venue_id",
        "venue_name",
        "venue_city",
        "venue_state",
        "is_active",
        "season",
        "ingested_at",
        "source_table",
    )
)
 
print(f"\nSample — first 5 rows:")
silver_df.show(5, truncate=False)

In [0]:
# ── WRITE TO SILVER ───────────────────────────────────────────────────────────
# First load (NHL): create table fresh.
# Subsequent loads (NFL, NBA etc.): MERGE on team_id + league to prevent dupes.
# Never overwrite — always append new leagues.
 
if not table_exists:
    # First run — create table
    (
        silver_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(SILVER_TABLE)
    )
    print(f"Table created: {SILVER_TABLE}")
 
elif LEAGUE in existing_leagues:
    # League already loaded — MERGE to pick up any changes (name updates etc.)
    silver_df.createOrReplaceTempView("new_teams")
 
    spark.sql(f"""
        MERGE INTO {SILVER_TABLE} AS target
        USING new_teams AS source
        ON target.team_id = source.team_id
        AND target.league = source.league
        WHEN MATCHED THEN UPDATE SET
            abbreviation    = source.abbreviation,
            display_name    = source.display_name,
            short_name      = source.short_name,
            name            = source.name,
            nickname        = source.nickname,
            location        = source.location,
            color           = source.color,
            alternate_color = source.alternate_color,
            logo_href       = source.logo_href,
            venue_id        = source.venue_id,
            venue_name      = source.venue_name,
            venue_city      = source.venue_city,
            venue_state     = source.venue_state,
            is_active       = source.is_active,
            season          = source.season,
            ingested_at     = source.ingested_at
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"Merged updates for existing league: {LEAGUE}")
 
else:
    # New league — append rows continuing the clutch_team_id sequence
    (
        silver_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(SILVER_TABLE)
    )
    print(f"Appended new league: {LEAGUE} (rows {max_id + 1}–{max_id + silver_df.count()})")

In [0]:
# ── VALIDATE ─────────────────────────────────────────────────────────────────
 
result = spark.sql(f"""
    SELECT
        clutch_team_id,
        team_id,
        sport,
        league,
        abbreviation,
        display_name,
        location,
        venue_name,
        is_active,
        season
    FROM {SILVER_TABLE}
    ORDER BY clutch_team_id
""")
 
total = result.count()
print(f"Total rows in {SILVER_TABLE}: {total}")
result.show(32, truncate=False)

In [0]:
# ── SANITY CHECKS ─────────────────────────────────────────────────────────────
 
checks = spark.sql(f"""
    SELECT
        COUNT(*)                                        AS total_teams,
        COUNT(DISTINCT league)                          AS leagues,
        COUNT(DISTINCT clutch_team_id)                  AS unique_clutch_ids,
        COUNT(CASE WHEN is_active = true  THEN 1 END)  AS active_teams,
        COUNT(CASE WHEN is_active = false THEN 1 END)  AS inactive_teams,
        COUNT(CASE WHEN display_name IS NULL THEN 1 END) AS null_names,
        MIN(clutch_team_id)                             AS min_id,
        MAX(clutch_team_id)                             AS max_id
    FROM {SILVER_TABLE}
""")
 
print("Sanity checks:")
checks.show(truncate=False)